# Chapter 13 Hands-On Coding Exam — Reproducible Notebook & Colab Engineering (100 Points)

**Book Reference:** Chapter 13, Section 13.7 (Hands-On Coding Exam)  
**Suggested Time:** 35–45 minutes (Open Book, Closed Solution)

| Task | Notebook & Colab Engineering Function | Points |
| :--- | :--- | :---: |
| **Task 13.1** | `audit_notebook_reproducibility(nb_path)` (`.ipynb` quality audit) | 30 |
| **Task 13.2** | `extract_code_to_module(nb_path, py_out_path)` (Notebook $\rightarrow$ `.py`) | 35 |
| **Task 13.3** | `resume_or_init_experiment(ckpt_path, default_state)` (Crash recovery) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q sharukh-tutor
import sharukh_tutor as tutor

tutor.activate(chapter="ch13", notebook="coding_exam")


---
## Exam Task 13.1 (30 Points): Audit Notebook Reproducibility

Write `audit_notebook_reproducibility(nb_path)` that loads a UTF-8 `.ipynb` JSON file and **returns** a dictionary with exact keys:
1. `"code_cells"`: integer count of cells where `cell_type == "code"`.
2. `"markdown_cells"`: integer count of cells where `cell_type == "markdown"`.
3. `"out_of_order"`: `True` if any code cell has `execution_count is None` or `<=` the previous executed code cell's count, else `False`.
4. `"has_outputs"`: `True` if any code cell has non-empty `outputs`, else `False`.
5. `"clean_for_git"`: `not has_outputs`.

In [ ]:
import json

def audit_notebook_reproducibility(nb_path):
    """Audit .ipynb cell counts, execution order, and Git cleanliness."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    code_cells = 0
    markdown_cells = 0
    out_of_order = False
    has_outputs = False
    prev = 0

    for cell in nb.get("cells", []):
        ctype = cell.get("cell_type")
        if ctype == "markdown":
            markdown_cells += 1
        elif ctype == "code":
            code_cells += 1
            ec = cell.get("execution_count")
            if ec is None or ec <= prev:
                out_of_order = True
            else:
                prev = ec
            if cell.get("outputs"):
                has_outputs = True

    return {
        "code_cells": code_cells,
        "markdown_cells": markdown_cells,
        "out_of_order": out_of_order,
        "has_outputs": has_outputs,
        "clean_for_git": not has_outputs,
    }
    ### END SOLUTION


---
## Exam Task 13.2 (35 Points): Extract Code Cells to Pure Python Module

Write `extract_code_to_module(nb_path, py_out_path)` that converts an exploratory notebook into a clean `.py` script:
1. Load `nb_path` as UTF-8 JSON and inspect only `cell_type == "code"` cells.
2. Drop any line whose stripped text starts with `!` (shell escape) or `%` (line/cell magic).
3. Join non-empty code cell blocks with blank lines (`"\n\n"`) and write the result in UTF-8 to `py_out_path`.
4. **Return** the integer count of non-empty Python code lines written.

In [ ]:
import json

def extract_code_to_module(nb_path, py_out_path):
    """Export code cells without !/% magics to .py and return line count."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    blocks = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = cell.get("source", [])
        text = "".join(src) if isinstance(src, list) else src
        kept = [
            line for line in text.splitlines()
            if not line.lstrip().startswith(("!", "%"))
        ]
        block = "\n".join(kept).strip("\n")
        if block:
            blocks.append(block)

    content = ("\n\n".join(blocks) + "\n") if blocks else ""
    with open(py_out_path, "w", encoding="utf-8") as out_f:
        out_f.write(content)
    return sum(1 for line in content.splitlines() if line.strip())
    ### END SOLUTION


---
## Exam Task 13.3 (35 Points): Resume or Atomically Initialize Experiment

Write `resume_or_init_experiment(ckpt_path, default_state)` to make long Colab training runs crash-proof:
1. If `ckpt_path` exists and parses as a valid JSON `dict`, load and **return** `(loaded_state, True)`.
2. Otherwise (missing file, corrupted/truncated JSON, or non-dict JSON), deep-copy `default_state`, atomically write it to `ckpt_path` via `.with_suffix(".tmp")` + `.replace(target)`, and **return** `(fresh_copy, False)`.

In [ ]:
import copy
import json
from pathlib import Path

def resume_or_init_experiment(ckpt_path, default_state):
    """Resume valid JSON dict checkpoint or atomically initialize it."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    target = Path(ckpt_path)
    if target.exists():
        try:
            with open(target, "r", encoding="utf-8") as f:
                loaded = json.load(f)
            if isinstance(loaded, dict):
                return loaded, True
        except (json.JSONDecodeError, OSError):
            pass

    fresh = copy.deepcopy(default_state)
    tmp_path = target.with_suffix(".tmp")
    with open(tmp_path, "w", encoding="utf-8") as out_f:
        json.dump(fresh, out_f, indent=2)
    tmp_path.replace(target)
    return fresh, False
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch13_exam_01": audit_notebook_reproducibility,
    "ch13_exam_02": extract_code_to_module,
    "ch13_exam_03": resume_or_init_experiment,
}

tutor.check_exam("ch13", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 13.1 (Resetting `prev` vs. Latching `out_of_order`):** Once a single code cell has `execution_count is None` or `<= prev`, `out_of_order` must stay `True`—do not overwrite it back to `False` on later cells. Also distinguish `out_of_order` (runtime order) from `clean_for_git` (`not has_outputs`).
- **Task 13.2 (Filtering Magics & Valid `.py` Syntax):** Leaving `!pip install` or `%matplotlib inline` inside an exported `.py` file immediately causes a `SyntaxError` when Python imports the module. Always check `line.lstrip().startswith(('!', '%'))` and ignore markdown cells.
- **Task 13.3 (Corrupted JSON Recovery & Deep Copy):** If Colab disconnects halfway through writing a checkpoint, `ckpt_path.exists()` is still `True` even though `json.load()` raises `JSONDecodeError`. Catching `json.JSONDecodeError`, checking `isinstance(loaded, dict)`, and deep-copying `default_state` guarantees safe recovery without mutating the caller's defaults.